# Official CODI KV spectral causality

Test whether the learned rank-four student key/value directions have causal value for answer accuracy beyond energy-matched random rank-four directions. The official CODI checkpoint is frozen. Position 4 and position 5 are the primary tests.

## 1. Choose the run scope

Keep the scientific constants unchanged. Run the smoke test once. The full run writes each condition atomically to Drive and can be resumed by rerunning its cell.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the printed immutable commit before the full run.
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"

RUN_SMOKE = True
RUN_FULL_EVALUATION = True
RUN_ANALYSIS = True

RANK = 4
RIDGE_RATIO = 1e-4
RANDOM_SEED = 20260727
POSITIONS = [0, 1, 2, 3, 4, 5]
PRIMARY_POSITIONS = [4, 5]
BATCH_SIZE = 128
MAX_NEW_TOKENS = 256
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0
FAMILYWISE_ALPHA = 0.05
PRECISION = "bfloat16"

## 2. Mount Drive and install the pinned environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import datetime
import json
import os
import pathlib
import subprocess
import sys
import time

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("Pin RUN_COMMIT before the full run to:", commit)

## 3. Verify the GPU and causal implementation

In [ ]:
import torch

assert torch.cuda.is_available(), "Select an A100 GPU runtime"
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_official_codi.py",
        "tests/test_kv_reduced_rank.py",
        "tests/test_official_codi_kv_intervention.py",
        "tests/test_official_codi_kv_causal_analysis.py",
    ],
    cwd=REPO_DIR,
    check=True,
)

## 4. Verify the official reproduction gate and calibration collection

This refuses to continue unless the official CODI checkpoint already passed the full 1,319-example GSM8K gate and the seed-1 calibration collection contains all 5,000 examples.

In [ ]:
from IPython.display import JSON, Markdown, display

drive_root = pathlib.Path(DRIVE_ROOT)
official_root = drive_root / "outputs" / "official_codi_gpt2"
reproduction_candidates = sorted(official_root.rglob("summary.json"))
REPRODUCTION_SUMMARY = None
for candidate in reproduction_candidates:
    payload = json.loads(candidate.read_text())
    gate = payload.get("gate")
    gate_status = gate.get("status") if isinstance(gate, dict) else gate
    count = payload.get("evaluated_counts", {}).get("gsm8k")
    if gate_status == "passed" and count == 1319:
        REPRODUCTION_SUMMARY = candidate
        break
assert REPRODUCTION_SUMMARY is not None, "Run the official CODI validation notebook first"

STATISTICS = drive_root / "outputs" / "official_codi_kv_subspaces" / "n5000_seed1" / "statistics.pt"
COLLECTION_MANIFEST = STATISTICS.parent / "collection_manifest.json"
assert STATISTICS.is_file(), f"Missing {STATISTICS}"
assert COLLECTION_MANIFEST.is_file(), f"Missing {COLLECTION_MANIFEST}"
collection = json.loads(COLLECTION_MANIFEST.read_text())
assert collection["state"] == "complete"
assert collection["processed_examples"] == 5000
print("Reproduction summary:", REPRODUCTION_SUMMARY)
print("Calibration statistics:", STATISTICS)
display(JSON({key: collection.get(key) for key in ("state", "processed_examples", "checkpoint_revision", "indices_sha256")}))

## 5. Export the compact learned and energy-matched random bases

The exporter fits student-side rank-four directions from the completed cross moments. It also records calibration means and groupwise random energy scales.

In [ ]:
OUTPUT_ROOT = drive_root / "outputs" / "official_codi_kv_causal"
REPORT_ROOT = drive_root / "reports" / "official_codi_kv_causal"
LOG_ROOT = drive_root / "logs" / "official_codi_kv_causal"
SUBSPACE_ARTIFACT = OUTPUT_ROOT / "subspaces" / "student_rank4.pt"
SUBSPACE_MANIFEST = SUBSPACE_ARTIFACT.with_suffix(".json")
for path in (SUBSPACE_ARTIFACT.parent, REPORT_ROOT, LOG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable, "scripts/export_official_codi_student_subspaces.py",
        "--statistics", str(STATISTICS),
        "--output", str(SUBSPACE_ARTIFACT),
        "--rank", str(RANK),
        "--ridge-ratio", str(RIDGE_RATIO),
        "--random-seed", str(RANDOM_SEED),
        "--minimum-examples", "5000",
    ],
    cwd=REPO_DIR,
    check=True,
)
subspace_manifest = json.loads(SUBSPACE_MANIFEST.read_text())
assert subspace_manifest["state"] == "complete"
assert subspace_manifest["rank"] == RANK
assert subspace_manifest["processed_examples"] == 5000
for kind in ("key", "value"):
    diagnostic = subspace_manifest["diagnostics"][kind]
    assert diagnostic["energy_match_max_relative_error"] < 1e-5
    assert diagnostic["learned_orthonormal_max_error"] < 1e-4
    assert diagnostic["random_orthonormal_max_error"] < 1e-4
display(JSON(subspace_manifest))

## 6. Persistent runner

Logs are appended to Drive. A successful command must return zero. The full evaluator itself resumes at the condition boundary.

In [ ]:
def run_persisted(command, log_name):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, command))} ===\n")
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return_code = process.wait()
        log.flush()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}; inspect {log_path}")
    return log_path

## 7. Smoke test positions 4 and 5

This exercises the real checkpoint, cache hook, artifact, and restart contract on 32 examples. It is diagnostic only.

In [ ]:
SMOKE_ROOT = OUTPUT_ROOT / "smoke_gsm8k"
if RUN_SMOKE:
    run_persisted(
        [
            sys.executable, "-u", "scripts/run_official_codi_kv_causal.py",
            "--config", "configs/official_codi_gpt2.yaml",
            "--reproduction-summary", str(REPRODUCTION_SUMMARY),
            "--subspace-artifact", str(SUBSPACE_ARTIFACT),
            "--output-dir", str(SMOKE_ROOT),
            "--positions", "4,5",
            "--rank", str(RANK),
            "--limit", "32",
            "--batch-size", "32",
            "--max-new-tokens", str(MAX_NEW_TOKENS),
            "--seed", str(RANDOM_SEED),
            "--precision", PRECISION,
            "--device", "cuda",
        ],
        "smoke_gsm8k.log",
    )
    smoke = json.loads((SMOKE_ROOT / "run_manifest.json").read_text())
    assert smoke["state"] == "complete"
    assert smoke["evaluated_count"] == 32
    assert len(smoke["conditions"]) == 9
    display(JSON(json.loads((SMOKE_ROOT / "summary.json").read_text())))
else:
    print("Smoke test skipped")

## 8. Run the complete full-GSM8K causal evaluation

This evaluates 29 conditions on all 1,319 GSM8K examples. Position 4 and position 5 run first. Rerun this cell after a disconnect. Verified completed conditions are skipped.

In [ ]:
FULL_ROOT = OUTPUT_ROOT / "full_gsm8k"
if RUN_FULL_EVALUATION:
    run_persisted(
        [
            sys.executable, "-u", "scripts/run_official_codi_kv_causal.py",
            "--config", "configs/official_codi_gpt2.yaml",
            "--reproduction-summary", str(REPRODUCTION_SUMMARY),
            "--subspace-artifact", str(SUBSPACE_ARTIFACT),
            "--output-dir", str(FULL_ROOT),
            "--positions", ",".join(map(str, POSITIONS)),
            "--include-all",
            "--rank", str(RANK),
            "--limit", "0",
            "--batch-size", str(BATCH_SIZE),
            "--max-new-tokens", str(MAX_NEW_TOKENS),
            "--seed", str(RANDOM_SEED),
            "--precision", PRECISION,
            "--device", "cuda",
        ],
        "full_gsm8k.log",
    )
    full_manifest = json.loads((FULL_ROOT / "run_manifest.json").read_text())
    assert full_manifest["state"] == "complete"
    assert full_manifest["evaluated_count"] == 1319
    assert len(full_manifest["conditions"]) == 29
    assert set(full_manifest["completed_conditions"]) == set(full_manifest["conditions"])
    print("All 29 full-GSM8K conditions are durable")
else:
    print("Full evaluation skipped")

## 9. Run the preregistered paired causal analysis

The primary family is retain and remove at positions 4 and 5. The analysis uses paired bootstrap intervals, exact McNemar tests, and Holm correction.

In [ ]:
REPORT_PATH = REPORT_ROOT / "official_codi_rank4_full_gsm8k.json"
if RUN_ANALYSIS:
    subprocess.run(
        [
            sys.executable, "scripts/analyze_official_codi_kv_causal.py",
            "--evaluation-root", str(FULL_ROOT),
            "--output", str(REPORT_PATH),
            "--primary-positions", ",".join(map(str, PRIMARY_POSITIONS)),
            "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
            "--seed", str(BOOTSTRAP_SEED),
            "--familywise-alpha", str(FAMILYWISE_ALPHA),
        ],
        cwd=REPO_DIR,
        check=True,
    )
    report = json.loads(REPORT_PATH.read_text())
    assert report["evaluated_count"] == 1319
    assert report["primary_positions"] == PRIMARY_POSITIONS
    display(Markdown(REPORT_PATH.with_suffix(".md").read_text()))
else:
    print("Analysis skipped")

## 10. Verify durable outputs

In [ ]:
print("\nDurable subspace artifact")
for path in sorted(SUBSPACE_ARTIFACT.parent.glob("*")):
    print(path.relative_to(OUTPUT_ROOT), f"{path.stat().st_size / 2**20:.1f} MiB")
print("\nDurable evaluation")
for path in sorted(FULL_ROOT.glob("*/summary.json")):
    payload = json.loads(path.read_text())
    print(path.parent.name, f"{payload['accuracy']:.4f}", payload["evaluated_count"])
print("\nDurable report")
for path in (REPORT_PATH, REPORT_PATH.with_suffix(".md")):
    print(path, f"{path.stat().st_size / 1024:.1f} KiB")
print("\nInterpretation boundary")
print(report["interpretation_boundary"])